<a href="https://colab.research.google.com/github/viki22uied/ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/viki22uied/ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Lane: Refresh / Content Opportunity Scoring**

**Answer 1 — what one row means.**
One row is one content item on one report date, for one client. It comes from
`fact_content_daily_performance`: `report_date × client_hash_id × content_hash_id`. A page shows up
once per day it has a search record, not once total.

**Answer 2 — which table or tables I will use.**
I will use `fact_content_daily_performance` (the daily search facts) joined to `dim_content` (static
content attributes like `content_created_date`, `keyword_token_count`, `search_volume`). I will not
need `dim_clients` or `fact_content_query_90d` for this contract.

**Answer 3 — which time window I will use.**
I will build on `month=2026-03` (March 2026), one partition of the daily fact table. I will not touch
June 2026 (`fact_content_daily_performance_sample`, the final month) for building or testing labels —
that window is sealed for later validation.

**Answer 4 — what I would predict or rank.**
A proxy label: *does this page need a content review?* I approximate "needs review" with low
click-through rate in the build window (bottom 30% of March CTR among pages with enough impressions).
This is a proxy, not a real outcome — a real refresh-worked label would need to observe what happened
after a page was actually refreshed, which this data does not record.

**Answer 5 — one thing I deliberately exclude, and why.**
I exclude `content_type` and `word_count` from `dim_content` as features. Both live in a table that
holds current state, not a point-in-time snapshot per month — `content_updated_date` for some rows in
this pull lands in June, after my March decision window. I cannot prove `content_type`/`word_count` held
the same value in March. Using them risks a quiet future leak, so I leave them out.

In [1]:
# Setup: connect DuckDB to the hosted warehouse. Never paste the token into a cell (public repo) —
# use an env var or Colab Secret named HF_TOKEN, prompt only as a last resort.
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

print('connected. build window: month=2026-03')


Paste your Hugging Face READ token (hf_...): ··········
connected. build window: month=2026-03


## 2. Fields: feature / label / context / excluded

- **Context** (grouping/joining only, never a model input): `client_hash_id`, `content_hash_id`.
  These are pseudonyms. They identify rows, they carry no signal.
- **Feature** (knowable by the end of March, safe to use — see Section 3 for the reason on each):
  `imp_march`, `avg_pos_march`, `content_age_days`, `keyword_token_count`, `search_volume`.
- **Label / proxy** (the thing I predict, never a feature): `ctr_march` (clicks ÷ impressions in
  March) feeds the `needs_review_label` proxy. `clk_march` (March clicks) is excluded from the
  feature list for the same reason — it is one half of the label's own formula.
- **Excluded** (kept out on purpose, with why):
  - `content_type`, `word_count` — from `dim_content`, a current-state table, not point-in-time.
    Some rows show `content_updated_date` after March, so I cannot confirm these held their March
    value. See Answer 5.
  - `gsc_avg_position` per-row — I use the March average, not the raw daily value, so a single
    noisy day cannot dominate.
  - Anything from `fact_content_daily_performance_sample` (June 2026) — sealed test month, per
    Answer 3.

In [2]:
# Print the bucket list so it is checked code, not just prose.
field_buckets = {
    'context': ['client_hash_id', 'content_hash_id'],
    'feature': ['imp_march', 'avg_pos_march', 'content_age_days', 'keyword_token_count', 'search_volume'],
    'label_or_proxy': ['ctr_march (-> needs_review_label)', 'clk_march (label input only)'],
    'excluded': ['content_type', 'word_count', 'raw daily gsc_avg_position', 'anything from June 2026 sample'],
}
for bucket, cols in field_buckets.items():
    print(f'{bucket}: {cols}')


context: ['client_hash_id', 'content_hash_id']
feature: ['imp_march', 'avg_pos_march', 'content_age_days', 'keyword_token_count', 'search_volume']
label_or_proxy: ['ctr_march (-> needs_review_label)', 'clk_march (label input only)']
excluded: ['content_type', 'word_count', 'raw daily gsc_avg_position', 'anything from June 2026 sample']


## 3. Verify it with queries (grain, counts, missing values, windows)

Three checks on `month=2026-03`, then a five-feature frame, then a deliberate leakage trap.

**Query 1 — prove the grain.** Group by `report_date, client_hash_id, content_hash_id` and look for
duplicates. Zero rows back means Answer 1 is true: one row really is one content item on one day for
one client.

In [3]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{MARCH}')
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f'duplicate grain rows found: {len(grain_check)}')
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,c


**Query 2 — row count and date span.** How many rows sit in my lane's slice, and does the date range
match the partition name.

In [4]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS n_content, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{MARCH}')
""").df()
span


,n_rows,min_date,max_date,n_content,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


**Query 3 — check availability.** `gsc_data_available` marks rows where GSC data actually exists for
that client on that day. Filter with `IS TRUE` (not `= TRUE`, which mishandles NULLs the same way,
but `IS TRUE` is explicit about intent) and count how many rows survive.

In [5]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM read_parquet('{MARCH}')
""").df()
availability['pct_available'] = (availability['gsc_available_rows'] / availability['total_rows'] * 100).round(1)
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,pct_available
0,9841378,3611061.0,36.7


### Five-feature frame, built from `month=2026-03` only

Decision moment: end of day, March 31, 2026. All five below are things I could actually know at
that moment.

1. **`imp_march`** (sum of GSC impressions in March) — Available at the decision moment because
   these are impressions Google already logged during March itself. By March 31 they are past, not
   future.
2. **`avg_pos_march`** (average GSC position in March) — Available at the decision moment because
   it is the average of daily positions already observed during March.
3. **`content_age_days`** (days between `content_created_date` and March 31) — Available at the
   decision moment because `content_created_date` is set once, at creation, and does not change
   afterward — no matter when I pull `dim_content`, this value is trustworthy as of March.
4. **`keyword_token_count`** (word count of the target keyword) — Available at the decision moment
   because it is fixed when the keyword record is created, before the page is even published.
5. **`search_volume`** (keyword's search demand estimate) — Available at the decision moment
   because it describes the keyword, not the page's performance — an external demand number that
   exists independent of anything the page did in March.

`clk_march` is deliberately left out of this feature list — it is a label input (see Section 2), not
a feature.

In [6]:
feats = con.sql(f"""
    WITH agg AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_march,
               SUM(gsc_clicks) AS clk_march,
               AVG(gsc_avg_position) AS avg_pos_march
        FROM read_parquet('{MARCH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1
        HAVING imp_march >= 50
    )
    SELECT a.content_hash_id, a.imp_march, a.clk_march, a.avg_pos_march,
           DATE_DIFF('day', d.content_created_date, DATE '2026-03-31') AS content_age_days,
           d.keyword_token_count,
           d.search_volume
    FROM agg a
    JOIN read_parquet('{DIM_CONTENT}') d USING (content_hash_id)
    WHERE d.content_created_date IS NOT NULL AND d.search_volume IS NOT NULL
""").df()

print(f'{len(feats):,} content items, {feats.isna().mean().mean():.0%} average missingness')
feats.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

113,552 content items, 0% average missingness


,content_hash_id,imp_march,clk_march,avg_pos_march,content_age_days,keyword_token_count,search_volume
0,content_39d7361b4945d504,77.0,0.0,4.074107,47,7,10
1,content_c03ecafd4c999f15,10849.0,22.0,8.240351,47,6,10
2,content_e689bc511192751a,61.0,0.0,6.015432,47,7,10
3,content_7dbc094b799e05a4,705.0,1.0,5.956862,47,10,10
4,content_40b10da45f4c1cb5,50.0,0.0,12.977513,47,8,10


### The trap

Label: `needs_review_label` = 1 if a page sits in the bottom 30% of March CTR. On purpose, I add
`ctr_march` (clicks ÷ impressions) as a sixth feature — the exact ratio the label is built from. I
train a small decision tree with it, then without it, and compare.

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feats['ctr_march'] = feats['clk_march'] / feats['imp_march']
threshold = feats['ctr_march'].quantile(0.30)
feats['needs_review_label'] = (feats['ctr_march'] <= threshold).astype(int)

feature_cols = ['imp_march', 'avg_pos_march', 'content_age_days', 'keyword_token_count', 'search_volume']
X = feats[feature_cols].copy()
y = feats['needs_review_label']

# add the leaked column on purpose
X_leaked = X.copy()
X_leaked['ctr_march'] = feats['ctr_march']

Xtr, Xte, ytr, yte = train_test_split(X_leaked, y, test_size=0.25, random_state=42, stratify=y)
leaked_model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xtr, ytr)
auc_leaked = roc_auc_score(yte, leaked_model.predict_proba(Xte)[:, 1])
print(f'LEAKED score (ctr_march included as a feature): AUC = {auc_leaked:.4f}')


LEAKED score (ctr_march included as a feature): AUC = 1.0000


In [8]:
# delete the leaked column and re-train
del X_leaked

Xtr2, Xte2, ytr2, yte2 = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xtr2, ytr2)
auc_honest = roc_auc_score(yte2, honest_model.predict_proba(Xte2)[:, 1])
print(f'HONEST score (5 features only, ctr_march removed): AUC = {auc_honest:.4f}')


HONEST score (5 features only, ctr_march removed): AUC = 0.8696


**Why the first score was fake.** `ctr_march` is not a signal that predicts the label — it IS the
label, restated as a number instead of a 0/1 flag. The tree just learned to re-read the threshold I
used to build the label in the first place. That is not a model, it is a lookup. The honest score,
with `ctr_march` removed, shows what the five real features can actually explain about CTR from
things known before the fact — clearly weaker, and that weakness is the true signal.

## 4. Data limits

**Named limitation: my slice is not the whole panel, and coverage is uneven.**

Only 55 of the warehouse's 104 clients have any row in March at all (Query 2), and of those rows,
only about 37% pass the `gsc_data_available IS TRUE` filter (Query 3). So my March feature frame
speaks for a slice of a slice — not the whole client base. A client missing from March is not a
client with zero search performance — it is a client whose GSC history had not started yet, or was
not connected. Any model trained on this slice learns from the clients who happened to have March
history, and that group may not look like the clients who don't.

In [9]:
clients_march = con.sql(f"""
    SELECT COUNT(DISTINCT client_hash_id) AS clients_with_march_rows
    FROM read_parquet('{MARCH}')
""").df()
total_clients = con.sql(f"""
    SELECT COUNT(*) AS total_clients
    FROM read_parquet('{REL}/dim_clients.parquet')
""").df()
print(f"clients with any March row: {clients_march['clients_with_march_rows'][0]} / {total_clients['total_clients'][0]}")


clients with any March row: 55 / 104


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
  (not committed yet — waiting on confirmation before committing)